# DQN results — every final table and figure

Reads the manifests produced by `10_dqn_suite_runner.ipynb` and turns them into
the numbers the thesis is written from. Nothing is trained here.

Every comparison against the original work uses `data/paper_ppo_*.csv`: the
authors' own 360 logged runs, extracted from their companion repository and
summarised with **our** metric. That means "the paper reports X" is a logged
return with a standard deviation, not a number read off a figure.

---
## 1. Setup

In [ ]:
import os, sys, subprocess, pathlib

GITHUB_USER, REPO_NAME, BRANCH = "RogerMas99", "qrl-dissection", "main"
try:
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = (f"https://{_tok}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if _tok
                else f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git")
except Exception:
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    CODE    = pathlib.Path("/content/qrl-dissection")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/results")
else:
    CODE    = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
    RESULTS = CODE / "results"
RES = RESULTS          # alias, so {RESULTS} in older cells still works
RESULTS.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    if CODE.exists():
        subprocess.run(["git", "-C", str(CODE), "pull", "--quiet"], check=False)
    else:
        subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, REPO_URL, str(CODE)], check=True)
    # SimplyQRL is vendored in this repo, so `pip install -e .` is the whole
    # install. No git dependency to resolve, which is what FIX-04 was about.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax", "jaxlib"], check=False)

sys.path.insert(0, str(CODE / "src"))
rev = subprocess.run(["git", "-C", str(CODE), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip()
print(f"code    {CODE} @ {rev}")
print(f"results {RESULTS}")

_need = False
if "autoray.autoray" in sys.modules:
    import autoray.autoray as _aa; _need = not hasattr(_aa, "NumpyMimic")
if "jax" in sys.modules: _need = True
if _need:
    print("\nIncompatible modules already loaded -> restarting runtime."); os.kill(os.getpid(), 9)
else:
    print("\nEnvironment clean. Continue below.")

# Fail loudly if the environment cell has not run. IPython substitutes {VAR} in
# `!` commands at run time and, when the name is undefined, passes the literal
# "{VAR}" to the shell instead of raising - so the error surfaces as bash
# complaining about a directory called "{CODE}". Better to stop here.
for _v in ("CODE", "RESULTS"):
    assert _v in dir(), f"{_v} is undefined - run the environment cell (section 1) first"

In [ ]:
import json, pathlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from qrl_dissection import analysis
from qrl_dissection.core import baselines as B

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})

def collect(subdir, window=50):
    """One row per completed cell, with the metric appropriate to its return."""
    d = RESULTS / subdir
    rows = []
    for mp in sorted(d.glob("*.manifest.json")):
        m = json.loads(mp.read_text())
        if "error" in m:
            continue
        oc = m.get("outcome", {})
        spec = m.get("spec", m)          # exp01/04 nest a spec; exp02/03 do not
        # Manifests store the ABSOLUTE path of the machine that produced them,
        # so a results folder moved between Drive layouts (or between machines)
        # breaks it. Fall back to where the file must be relative to this
        # directory - SafeDQN always writes runs/<run_name>.csv inside outdir.
        name = m.get("run_name") or m.get("name") or mp.stem.replace(".manifest", "")
        csv = oc.get("episodes_csv")
        if not csv or not pathlib.Path(csv).exists():
            csv = str(d / "runs" / f"{name}.csv")
        if not pathlib.Path(csv).exists():
            continue
        rew, step = analysis.load_episodes(csv)
        greedy = np.nan
        ev = oc.get("eval_csv") or ""
        if not pathlib.Path(ev).exists():
            ev = str(d / "runs" / f"{name}_eval.csv")
        if pathlib.Path(ev).exists():
            _s, sc = analysis.load_eval(ev)
            greedy = float(max(sc)) if len(sc) else np.nan
        rows.append(dict(
            name=name,
            arm=spec.get("arm", m.get("arm", "?")),
            seed=spec.get("seed", m.get("seed")),
            fix01=spec.get("fix_autoreset", m.get("fix_autoreset")),
            best_ma=float(np.nanmax(analysis.moving_average(rew, window))),
            greedy_best=greedy,
            phantom=oc.get("probe", {}).get("frac_poison"),
            csv=csv))
    return pd.DataFrame(rows)

def curve(csv, window=50):
    rew, step = analysis.load_episodes(csv)
    return step, analysis.moving_average(rew, window)

def band(ax, frame, label, color, window=50):
    """Mean +/- sd across seeds on a common step grid."""
    grid = np.linspace(0, 100_000, 200)
    stack = []
    for c_ in frame.csv:
        s, y = curve(c_, window)
        stack.append(np.interp(grid, s, np.nan_to_num(y, nan=np.nanmin(y))))
    if not stack: return
    arr = np.vstack(stack); mu, sd = arr.mean(0), arr.std(0)
    ax.plot(grid, mu, color=color, label=f"{label} (n={len(stack)})")
    ax.fill_between(grid, mu - sd, mu + sd, color=color, alpha=.15)

print("helpers ready")

---
## 2. What is actually finished

Seed counts first, every time. A table read without them invites conclusions the
data cannot carry.

In [ ]:
for sub in ["exp01_dqn_cartpole_capacity", "exp02_dqn_cartpole_output_reuse",
            "exp03_dqn_cartpole_data_reuploading",
            "exp03b_dqn_cartpole_dr_unentangled",
            "exp04_dqn_frozenlake_embeddings"]:
    df = collect(sub)
    n = df.seed.nunique() if len(df) else 0
    flag = "" if n >= 10 else ("  <- COVERAGE ONLY, not a conclusion" if n else "  <- empty")
    print(f"{sub:42s} {len(df):3d} cells, {n:2d} seeds{flag}")

---
## 2b. Report intervals, not point estimates

Following Agarwal et al., *Deep RL at the Edge of the Statistical Precipice*
(NeurIPS 2021). Three changes, and one of them matters more than the other two.

**The metric was biased.** `best_ma50` and `greedy_best` are *maxima over
training*. A maximum over a noisy curve is positively biased, and the bias grows
with variance — so a noisier arm scores higher for nothing. Demonstrated in
`tests/test_stats.py`: two arms with an identical true mean of 200, differing
only in variance, score 567 and 306 under a max-over-training protocol. That is
an invented 85% advantage.

This is not academic here. The paper's own OR block has quantum arms at ±115 and
classical arms at ±37 — the quantum side is three times noisier, so the protocol
systematically flatters it. Agarwal et al. identify exactly this class of
non-standard evaluation protocol as an explanation for apparent gains elsewhere.

`final_performance` (mean over the last 10% of episodes) is the unbiased
counterpart, and it comes out of CSVs you already have. No retraining.

**IQM instead of mean.** The mean of the middle 50% of runs. Keeps the median's
robustness against a diverging run — which VQ-DQN produces regularly, per Franz
et al. (2022) — with much less uncertainty.

**Bootstrap CIs instead of ±sd.** Agarwal et al. validate percentile CIs at
N = 10 and warn that at N = 3 they *underestimate* the true interval. That is the
statistical argument for plan B, and it is a better one than "more seeds is
nicer".

In [ ]:
from qrl_dissection.core import stats as S

def scores(subdir, metric="final", last_frac=0.1, window=50):
    """Per-seed scores for one experiment, grouped by arm.

    metric="final"  mean over the last 10% of episodes  (unbiased, recommended)
    metric="max"    best moving average over training   (biased, for comparison)
    """
    out = {}
    for _, r in collect(subdir, window).iterrows():
        rew, _ = analysis.load_episodes(r.csv)
        v = (S.final_performance(rew, last_frac) if metric == "final"
             else float(np.nanmax(analysis.moving_average(rew, window))))
        out.setdefault(r.arm, []).append(v)
    return out

def report(by_arm, title=""):
    rows = [dict(arm=k, **S.summarise_scores(v)) for k, v in sorted(by_arm.items())]
    df = pd.DataFrame(rows)[["arm", "n", "iqm", "ci_low", "ci_high", "median", "mean"]]
    if title: print(title)
    display(df.round(2))
    if any(r["n"] < 10 for r in rows):
        print("!! n < 10: the bootstrap CI is narrower than the true 95% interval "
              "(Agarwal et al., Fig. 6). Report n beside every interval.")
    return df

# Side by side: what the biased metric claims, and what the unbiased one shows.
for sub in ["exp03_dqn_cartpole_data_reuploading",
            "exp03b_dqn_cartpole_dr_unentangled"]:
    try:
        report(scores(sub, "max"),   f"=== {sub} - MAX over training (biased) ===")
        report(scores(sub, "final"), f"=== {sub} - final 10% (unbiased) ===")
    except Exception as exc:
        print(f"{sub}: {exc}")

In [ ]:
# The honest phrasing of a comparison at small N. "0.62" means a randomly chosen
# run of A beats a randomly chosen run of B 62% of the time - a weaker and more
# accurate claim than "A is better than B", and it does not pretend two means are
# separated when the runs overlap.
a = scores("exp03_dqn_cartpole_data_reuploading", "final")
b = scores("exp03b_dqn_cartpole_dr_unentangled", "final")
for arm in sorted(set(a) & set(b)):
    p = S.probability_of_improvement(a[arm], b[arm])
    print(f"  {arm:28s} P(exp03 run > exp03b run) = {p:.2f}"
          f"   (n={len(a[arm])} vs {len(b[arm])})")

---
## 3. exp03 vs exp03b — what was Data Reuploading actually doing?

The most important figure in the DQN half.

exp03 swept depth with `ent=True`. FIX-07 showed the `skolik` template's final
entangling ring cannot affect a PauliZ readout, so depths 1/2/5 carried 0/1/4
*effective* entangling blocks — the depth axis moved entanglement too. exp03b is
the same grid with `ent=False`, where every depth has zero.

If the two curves rise together, exp03's claim is about reuploading and it
survives. If exp03b flattens, the claim was about entanglement and
`RESULTS-LOG.md` needs rewriting rather than annotating.

In [ ]:
a = collect("exp03_dqn_cartpole_data_reuploading")
b = collect("exp03b_dqn_cartpole_dr_unentangled")

def depth_of(name):
    import re
    m = re.search(r"DR(\d+)", str(name))
    return int(m.group(1)) if m else np.nan

for f in (a, b):
    if len(f): f["depth"] = f.name.map(depth_of)

tab = []
for label, f in (("exp03 ent=True", a), ("exp03b ent=False", b)):
    if not len(f): continue
    g = f.groupby("depth").agg(greedy=("greedy_best","mean"), sd=("greedy_best","std"),
                               best_ma=("best_ma","mean"), n=("seed","count"))
    g["experiment"] = label
    tab.append(g.reset_index())
if tab:
    display(pd.concat(tab).set_index(["experiment","depth"]).round(1))

fig, ax = plt.subplots(figsize=(7, 4.2))
for label, f, style in (("exp03  ent=True", a, "o-"), ("exp03b ent=False", b, "s--")):
    if not len(f): continue
    g = f.groupby("depth").greedy_best.agg(["mean","std"])
    ax.errorbar(g.index, g["mean"], yerr=g["std"], fmt=style, capsize=3, label=label)
p = B.paper_dr_curve("Skolik_8Q")
ax.errorbar([d for d,_m,_s in p], [m for _d,m,_s in p], yerr=[s for _d,_m,s in p],
            fmt="^:", capsize=3, color="gray", label="paper, PPO, n=10")
ax.set_xlabel("Data Reuploading depth (n_layers_q)")
ax.set_ylabel("greedy evaluation return")
ax.set_title("Does DR transfer to DQN, and is it DR that transfers?")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

---
## 4. exp01 — the circuit against a fair control

The control sees the **full** observation and carries at least the hybrid's
parameter count. Both halves matter: the paper's classical arm discards cart
position, which is one of CartPole's two termination conditions, so an arm
without it can fail from blindness rather than from being classical (NEW-02).

In [ ]:
df = collect("exp01_dqn_cartpole_capacity")
if len(df):
    display(df.groupby(["arm","fix01"]).agg(
        greedy=("greedy_best","mean"), sd=("greedy_best","std"),
        best_ma=("best_ma","mean"), phantom=("phantom","mean"),
        n=("seed","count")).round(1))

    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for (arm, color) in zip(sorted(df.arm.unique()),
                            ["tab:blue","tab:orange","tab:green","tab:red","tab:purple"]):
        band(ax, df[(df.arm==arm) & (df.fix01==True)], arm, color)
    ax.axhline(22, ls=":", c="gray", lw=.9, label="random policy (~22)")
    ax.set_xlabel("step"); ax.set_ylabel("episodic return (MA-50)")
    ax.set_title("exp01 - CartPole, FIX-01 on"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("exp01 not run yet")

---
## 5. exp02 — Output Reuse, against the paper's own OR block

The paper's OR numbers are now available at 10 seeds for both arms, so this is a
real comparison rather than a gesture at a figure. Read the spread: their
`Quantum_r32` is 430.5 ± 115.6 against `Classical_r16` at 381.7 ± 37.3 — the
quantum advantage is about a third of its own standard deviation, which is not a
separation.

In [ ]:
df = collect("exp02_dqn_cartpole_output_reuse")
if len(df):
    import re
    df["R"] = df.name.map(lambda n: int(re.search(r"[rR](\d+)", str(n)).group(1))
                          if re.search(r"[rR](\d+)", str(n)) else np.nan)
    display(df.groupby("R").agg(greedy=("greedy_best","mean"), sd=("greedy_best","std"),
                                n=("seed","count")).round(1))
    g = df.groupby("R").greedy_best.agg(["mean","std"])
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.errorbar(g.index, g["mean"], yerr=g["std"], fmt="o-", capsize=3, label="ours, DQN")
    s = B.load_paper_summary()
    for pref, style, col in (("Quantum_r","^:","gray"), ("Classical_r","v:","black")):
        rows = s[s.config.str.startswith(pref)].copy()
        rows["R"] = rows.config.str.replace(pref, "", regex=False).astype(int)
        rows = rows.sort_values("R")
        ax.errorbar(rows.R, rows.best_ma50_mean, yerr=rows.best_ma50_sd,
                    fmt=style, capsize=3, color=col, label=f"paper {pref[:-2]}, PPO, n=10")
    ax.set_xscale("log", base=2); ax.set_xlabel("Output Reuse factor R")
    ax.set_ylabel("return"); ax.legend(fontsize=8)
    ax.set_title("exp02 - Output Reuse"); plt.tight_layout(); plt.show()
else:
    print("exp02 not run yet")

---
## 6. exp04 — FrozenLake

The return is a single bit, so a rolling mean **is** the success rate. A
100-episode window rather than 50: on a binary signal the shorter one is noisy.

This is also where FIX-01 becomes measurable. On CartPole the phantom fraction
shrinks as the agent improves — under 1% once an arm learns — which is why exp01
found no significant effect in any live arm. On FrozenLake it stays put:
**measured 13.2% at 100k steps**, against a random-policy prediction of 13.0%.

In [ ]:
df = collect("exp04_dqn_frozenlake_embeddings", window=100)
if len(df):
    display(df.groupby(["arm","fix01"]).agg(
        success=("best_ma","mean"), sd=("best_ma","std"), greedy=("greedy_best","mean"),
        phantom=("phantom","mean"), n=("seed","count")).round(3))

    print("\n=== H3: FIX-01 where the phantom fraction is ~10x CartPole's ===")
    for arm in sorted(df.arm.unique()):
        on  = df[(df.arm==arm) & (df.fix01==True)].best_ma
        off = df[(df.arm==arm) & (df.fix01==False)].best_ma
        if len(on) and len(off):
            pooled = np.sqrt((on.std()**2 + off.std()**2)/2)
            print(f"  {arm:26s} delta {on.mean()-off.mean():+.3f}  "
                  f"pooled sd {pooled:.3f}  n={len(off)}+{len(on)}")
    print("  CartPole reference (exp01): no significant FIX-01 effect in any live arm.")

    live = "frozen_onehot_mlp"
    if live in set(df.arm):
        best = df[df.arm==live].best_ma.max()
        print(f"\nLIVENESS GATE: {live} best success {best:.2f}",
              "- PASS" if best > 0.9 else "- FAIL, stage 2 is uninterpretable")
else:
    print("exp04 not run yet")

---
## 7. Export

Figures and the summary tables to Drive, so the thesis can cite files rather than
notebook scrollback. Then copy the tables into `docs/RESULTS-LOG.md` and commit —
a number that exists only in a notebook output is a number that will be lost.

In [ ]:
OUT = RESULTS / "_figures"; OUT.mkdir(exist_ok=True)
frames = {}
for sub in ["exp01_dqn_cartpole_capacity", "exp02_dqn_cartpole_output_reuse",
            "exp03_dqn_cartpole_data_reuploading",
            "exp03b_dqn_cartpole_dr_unentangled",
            "exp04_dqn_frozenlake_embeddings"]:
    f = collect(sub)
    if len(f):
        f.drop(columns=["csv"]).to_csv(OUT / f"{sub}_summary.csv", index=False)
        frames[sub] = f
print("exported:", *[p.name for p in sorted(OUT.glob('*.csv'))], sep="\n  ")
print(f"\nSeed counts - the first thing a reader should see:")
for k, f in frames.items():
    print(f"  {k:42s} {f.seed.nunique():2d} seeds")